In [ ]:
# ==============================================================
# 05 – Feature Fusion (Tabular + CNN Embeddings + Country)
# Creates the final state representation for DRL / MARL agents
# Supports RQ1 and RQ2
# ==============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
%matplotlib inline

ROOT = Path(".")
DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

print("Feature Fusion Notebook – Multi-Country Version")

# --------------------------------------------------------------
# 1. Load all available data sources
# --------------------------------------------------------------
# A. Meta / Tabular data
meta_path = DATA_SYNTHETIC / "global_credit_from_german.csv"
df = pd.read_csv(meta_path)
print(f"Meta data shape: {df.shape}")

# B. CNN Embeddings (from notebook 03)
emb_path = DATA_PROCESSED / "cnn_embeddings.npy"
if emb_path.exists():
    cnn_emb = np.load(emb_path)
    print(f"CNN embeddings shape: {cnn_emb.shape}")
else:
    raise FileNotFoundError("CNN embeddings not found. Please run 03_CNN_Encoder.ipynb first.")

assert len(df) == len(cnn_emb), "Mismatch between meta data and CNN embeddings!"

y = df["default"].values.astype(np.int64)
thin = df["thin_file"].values.astype(np.int64)

# --------------------------------------------------------------
# 2. Prepare Tabular Features
# --------------------------------------------------------------
# Numerical features
num_candidates = ['duration', 'credit_amount', 'age', 'installment_rate',
                  'residence_since', 'existing_credits', 'num_dependents', 'income_proxy']
num_cols = [c for c in num_candidates if c in df.columns]

# Categorical features (simple label encoding)
cat_candidates = ['checking_status', 'credit_history', 'purpose', 'savings',
                  'employment', 'property', 'housing', 'job']
cat_cols = [c for c in cat_candidates if c in df.columns]

tabular = df[num_cols].copy()

for col in cat_cols:
    le = LabelEncoder()
    tabular[col] = le.fit_transform(df[col].astype(str))

# Thin-file flag
tabular['thin_file'] = thin

print(f"Tabular features shape (before scaling): {tabular.shape}")

# --------------------------------------------------------------
# 3. Country Embedding (simple one-hot + optional learned later)
# --------------------------------------------------------------
if 'country' in df.columns:
    country_dummies = pd.get_dummies(df['country'], prefix='country')
    tabular = pd.concat([tabular, country_dummies], axis=1)
    print(f"Added country dummies. New shape: {tabular.shape}")

# --------------------------------------------------------------
# 4. Scale Tabular Features
# --------------------------------------------------------------
scaler_tab = StandardScaler()
tabular_scaled = scaler_tab.fit_transform(tabular).astype(np.float32)

# --------------------------------------------------------------
# 5. Fuse Features
# --------------------------------------------------------------
# Version 1: Full fusion (Tabular + CNN)
X_fused = np.concatenate([tabular_scaled, cnn_emb], axis=1)
print(f"\nFused feature matrix shape: {X_fused.shape}")

# Version 2: PCA-reduced fusion (optional – useful for faster MARL)
pca = PCA(n_components=min(64, X_fused.shape[1]), random_state=42)
X_fused_pca = pca.fit_transform(X_fused).astype(np.float32)
print(f"PCA-reduced fused shape: {X_fused_pca.shape}")
print(f"Explained variance by PCA: {pca.explained_variance_ratio_.sum():.2%}")

# --------------------------------------------------------------
# 6. Train / Test Split (stratified)
# --------------------------------------------------------------
X_train, X_test, y_train, y_test, thin_train, thin_test = train_test_split(
    X_fused, y, thin, test_size=0.25, random_state=42, stratify=y
)

X_train_pca, X_test_pca = train_test_split(
    X_fused_pca, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTrain fused shape: {X_train.shape}")
print(f"Test  fused shape: {X_test.shape}")

# --------------------------------------------------------------
# 7. Save everything (Production ready)
# --------------------------------------------------------------
np.save(DATA_PROCESSED / "X_fused.npy", X_fused)
np.save(DATA_PROCESSED / "X_fused_pca.npy", X_fused_pca)
np.save(DATA_PROCESSED / "X_train_fused.npy", X_train)
np.save(DATA_PROCESSED / "X_test_fused.npy", X_test)
np.save(DATA_PROCESSED / "X_train_pca.npy", X_train_pca)
np.save(DATA_PROCESSED / "X_test_pca.npy", X_test_pca)

np.save(DATA_PROCESSED / "y.npy", y)
np.save(DATA_PROCESSED / "y_train.npy", y_train)
np.save(DATA_PROCESSED / "y_test.npy", y_test)
np.save(DATA_PROCESSED / "thin.npy", thin)
np.save(DATA_PROCESSED / "thin_train.npy", thin_train)
np.save(DATA_PROCESSED / "thin_test.npy", thin_test)

joblib.dump(scaler_tab, DATA_PROCESSED / "scaler_tabular.joblib")
joblib.dump(pca, DATA_PROCESSED / "pca_fusion.joblib")
joblib.dump(list(tabular.columns), DATA_PROCESSED / "tabular_feature_names.joblib")

print("\n✓ All fused features saved to data/processed/")

# --------------------------------------------------------------
# 8. Quick Visualization
# --------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PCA variance
axes[0].plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
axes[0].set_title("PCA Cumulative Explained Variance")
axes[0].set_xlabel("Number of Components")
axes[0].set_ylabel("Cumulative Variance")
axes[0].grid(True)

# Class distribution in fused space (first 2 PCA components)
scatter = axes[1].scatter(X_fused_pca[:, 0], X_fused_pca[:, 1], c=y, cmap="coolwarm", alpha=0.5, s=10)
axes[1].set_title("Fused Features (PCA) – Colored by Default")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
plt.colorbar(scatter, ax=axes[1], label="Default")

plt.tight_layout()
plt.savefig(RESULTS / "feature_fusion_overview.png", dpi=140, bbox_inches="tight")
plt.show()

print("\n✅ Feature Fusion completed successfully.")
print("You can now proceed to:")
print("  → 06_Single_Agent_PPO.ipynb")
print("  → 07_Multi_Agent_Training.ipynb")